# 5.2 GEMM：`acl.blas.gemm_ex`

## 小节概述

目标：先用 ATC 编译 GEMM 单算子 OM，再通过 `acl.blas.gemm_ex` 提交 GEMM，读回结果并与 NumPy reference 比较。

<table style="text-align:left; margin-left:0;">
<tr><th>参数/对象</th><th>本实验含义</th></tr>
<tr><td><code>M/N/K</code></td><td>输出行数、输出列数、归约维</td></tr>
<tr><td><code>float16</code></td><td>A、B、C、alpha、beta 的存储类型</td></tr>
<tr><td><code>ACL_COMPUTE_HIGH_PRECISION</code></td><td><code>gemm_ex</code> 的计算精度策略</td></tr>
<tr><td>Stream</td><td>异步提交算子；必须同步后才能 D2H</td></tr>
<tr><td>Event</td><td>记录同一 Stream 上重复算子的设备时间，不包含 ATC、H2D、D2H</td></tr>
<tr><td>模型目录</td><td>ATC 单算子 OM 所在目录，由 <code>acl.op.set_model_dir</code> 注册</td></tr>
</table>


In [ ]:
import json
import subprocess
import time
from pathlib import Path

import acl
import numpy as np

SOC_VERSION = acl.get_soc_name()

M, N, K = 64, 64, 64
DEVICE_ID = 0
REPEAT = 10
MODEL_DIR = (Path("work") / "01.02_gemm" / "models").resolve()
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"M={M}, N={N}, K={K}, dtype=float16, device={DEVICE_ID}, repeat={REPEAT}, soc={SOC_VERSION}")


## 生成并编译单算子描述

ATC 把算子描述编译为适配当前 SoC（`SOC_VERSION`）的 OM。后续 `acl.op.set_model_dir` 只注册该目录，不加载整图模型。


In [ ]:
MODEL_DIR = (Path("work") / "01.02_gemm" / "models").resolve()
MODEL_DIR.mkdir(parents=True, exist_ok=True)

gemm_singleop = [{
    "op": "GEMM",
    "input_desc": [
        {"format": "ND", "shape": [M, K], "type": "float16"},  # a
        {"format": "ND", "shape": [K, N], "type": "float16"},  # b
        {"format": "ND", "shape": [M, N], "type": "float16"},  # c（输出就地写入）
        {"format": "ND", "shape": [], "type": "float16"},      # alpha：标量
        {"format": "ND", "shape": [], "type": "float16"},      # beta：标量
    ],
    "output_desc": [{"format": "ND", "shape": [M, N], "type": "float16"}],
    "attr": [
        {"name": "transpose_a", "type": "bool", "value": False},
        {"name": "transpose_b", "type": "bool", "value": False},
    ],
}]

json_path = MODEL_DIR / "gemm_singleop.json"
json_path.write_text(json.dumps(gemm_singleop, indent=2, ensure_ascii=False), encoding="utf-8")
print("已写入", json_path)


In [ ]:
cmd = [
    "atc", f"--singleop={json_path}", f"--output={MODEL_DIR}",
    f"--soc_version={SOC_VERSION}",
]
print("$", *cmd)
subprocess.run(cmd, check=True)


## 直接调用 GEMM 并校验

主线只包含设备数据准备、`acl.blas.gemm_ex`、Event 计时、同步、D2H、NumPy 校验和逆序释放。输出报告分别记录实际调用入口、Device、平均设备时间、CPU reference 时间和最大误差；不能从 API 名称推断具体 Kernel 类别。


In [ ]:
# pyACL 使用整数常量表示数据类型、拷贝方向和内存策略。
ACL_FLOAT16 = 1
ACL_COMPUTE_HIGH_PRECISION = 0
MEMCPY_H2D = 1
MEMCPY_D2H = 2
MALLOC_FLAG = 2


def check(ret, call):
    if ret != 0:
        raise RuntimeError(f"{call} 失败，ret={ret}")


a_host = ((np.arange(M)[:, None] * 3 + np.arange(K)[None, :]) % 7).astype(np.float16)
b_host = ((np.arange(K)[:, None] * 2 + np.arange(N)[None, :]) % 9).astype(np.float16)
c_host = np.zeros((M, N), dtype=np.float16)
alpha_host = np.array([1.0], dtype=np.float16)
beta_host = np.array([0.0], dtype=np.float16)

context = stream = start_event = end_event = None
acl_inited = device_set = False
ptrs = []


def h2d(array):
    ptr, ret = acl.rt.malloc(array.nbytes, MALLOC_FLAG)
    check(ret, "acl.rt.malloc")
    ptrs.append(ptr)
    check(acl.rt.memcpy(
        ptr, array.nbytes, acl.util.numpy_to_ptr(array), array.nbytes, MEMCPY_H2D
    ), "acl.rt.memcpy H2D")
    return ptr


try:
    check(acl.init(), "acl.init")
    acl_inited = True
    check(acl.op.set_model_dir(str(MODEL_DIR)), "acl.op.set_model_dir")
    check(acl.rt.set_device(DEVICE_ID), "acl.rt.set_device")
    device_set = True
    context, ret = acl.rt.create_context(DEVICE_ID)
    check(ret, "acl.rt.create_context")
    stream, ret = acl.rt.create_stream()
    check(ret, "acl.rt.create_stream")
    start_event, ret = acl.rt.create_event()
    check(ret, "acl.rt.create_event start")
    end_event, ret = acl.rt.create_event()
    check(ret, "acl.rt.create_event end")

    current_device, ret = acl.rt.get_device()
    check(ret, "acl.rt.get_device")
    dev_a = h2d(a_host)
    dev_b = h2d(b_host)
    dev_c = h2d(c_host)
    dev_alpha = h2d(alpha_host)
    dev_beta = h2d(beta_host)

    check(acl.rt.record_event(start_event, stream), "acl.rt.record_event start")
    for _ in range(REPEAT):
        check(acl.blas.gemm_ex(
            0, 0, 0,
            M, N, K,
            dev_alpha, dev_a, -1, ACL_FLOAT16,
            dev_b, -1, ACL_FLOAT16,
            dev_beta, dev_c, -1, ACL_FLOAT16,
            ACL_COMPUTE_HIGH_PRECISION, stream,
        ), "acl.blas.gemm_ex")
    check(acl.rt.record_event(end_event, stream), "acl.rt.record_event end")
    check(acl.rt.synchronize_stream(stream), "acl.rt.synchronize_stream")
    elapsed_ms, ret = acl.rt.event_elapsed_time(start_event, end_event)
    check(ret, "acl.rt.event_elapsed_time")

    out_host = np.empty((M, N), dtype=np.float16)
    check(acl.rt.memcpy(
        acl.util.numpy_to_ptr(out_host), out_host.nbytes,
        dev_c, c_host.nbytes, MEMCPY_D2H,
    ), "acl.rt.memcpy D2H")

    reference_start = time.perf_counter()
    golden = (a_host.astype(np.float32) @ b_host.astype(np.float32)).astype(np.float16)
    reference_ms = (time.perf_counter() - reference_start) * 1000.0
    max_abs_error = float(np.max(np.abs(out_host.astype(np.float32) - golden.astype(np.float32))))
    np.testing.assert_allclose(out_host, golden, atol=0.02, rtol=0)

    gemm_result = {
        "status": "PASS",
        "actual_backend": "pyACL/acl.blas.gemm_ex device path; kernel class not inferred",
        "device_id": int(current_device),
        "shape": [M, N],
        "dtype": "float16",
        "repeat": REPEAT,
        "device_mean_ms": elapsed_ms / REPEAT,
        "reference_ms": reference_ms,
        "max_abs_error": max_abs_error,
        "fallback": 0,
    }
    print("GEMM_REPORT=" + json.dumps(gemm_result, ensure_ascii=False))
finally:
    for event in (end_event, start_event):
        if event is not None:
            acl.rt.destroy_event(event)
    for ptr in reversed(ptrs):
        acl.rt.free(ptr)
    if stream is not None:
        acl.rt.destroy_stream(stream)
    if context is not None:
        acl.rt.destroy_context(context)
    if device_set:
        acl.rt.reset_device(DEVICE_ID)
    if acl_inited:
        acl.finalize()


## 课后实践

把 `M、N、K` 改为 `32、64、128`，同步修改单算子描述，使用独立输入重新运行。记录：

1. A、B、输出 shape 与 M/N/K 的对应关系；
2. `actual_backend`、Device ID、平均设备时间和最大误差；
3. 为什么 Event 计时不包含 ATC、H2D 和 D2H。

下面的 Cell 只准备独立输入，不替代真实设备调用。完成后按 1.2 主链提交并校验结果。


In [ ]:
PRACTICE_M, PRACTICE_N, PRACTICE_K = 32, 64, 128
practice_a = ((np.arange(PRACTICE_M)[:, None] + np.arange(PRACTICE_K)[None, :]) % 7).astype(np.float16)
practice_b = ((np.arange(PRACTICE_K)[:, None] * 2 + np.arange(PRACTICE_N)[None, :]) % 9).astype(np.float16)
practice_golden = (practice_a.astype(np.float32) @ practice_b.astype(np.float32)).astype(np.float16)
assert practice_golden.shape == (PRACTICE_M, PRACTICE_N)
print("独立 GEMM 输入已准备：", practice_a.shape, practice_b.shape, practice_golden.shape)


In [ ]:
# 完成练习后按需执行；Notebook 不会自动展开答案。
!cat answer/05.02_gemm_answer.md
